# 02: Preprocessing Pipeline (FastICA & Patch Extraction)

This notebook demonstrates the preprocessing pipeline used to transform the raw FENIX hyperspectral cubes into spatial-spectral patches ready for the Multi-Scale CNN.

The pipeline consists of four main steps:
1. **Normalization**: Standardizing the 450 spectral bands.
2. **Dimensionality Reduction (FastICA)**: Reducing the 450 bands to 30 independent components to isolate distinct mineral signatures while discarding noise.
3. **Patch Extraction**: Slicing the core into overlapping 25x25 spatial patches.
4. **Stratified Splitting**: Creating Train/Validation/Test splits while maintaining the class distribution of rare minerals.

In [ ]:
# 1. Setup Environment (Colab)
!git clone https://github.com/suva1998/hsi-lithology-showcase.git
%cd hsi-lithology-showcase
!pip install -e .

In [ ]:
# Load Configuration
import yaml
from pprint import pprint

with open('configs/mla_drillcore.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Preprocessing Configuration:")
pprint(config['preprocessing'])

In [ ]:
# 2. Download Data
from src.data.download import download_dataset
import numpy as np

# This will download the dataset if not present, and load all 44 Stonepark cores
data, labels = download_dataset("mla_drillcore", config['paths']['data_dir'])

print(f"\nLoaded Data Shape: {data.shape} (Pixels, Width, Bands)")
print(f"Loaded Labels Shape: {labels.shape}")

## Run Preprocessing Pipeline

The `preprocess_hsi` function orchestrates the entire pipeline. It takes the stacked array of cores and returns a dictionary containing the generated data splits, the fitted Normalizer and FastICA reducers, an RGB composite of the ICA components, and class counts.

In [ ]:
from src.data.preprocessing import preprocess_hsi
import time

start_time = time.time()
print("Starting preprocessing pipeline...")

# This may take a few minutes depending on the CPU, as FastICA is computationally intensive
result = preprocess_hsi(data, labels, config)

print(f"\nPreprocessing completed in {time.time() - start_time:.2f} seconds!")


## Visualize the Results

Let's inspect the shapes of the generated splits and visualize the ICA false-color composite.

In [ ]:
import matplotlib.pyplot as plt

print("Dataset Splits:")
for split_name, (x, y) in result['splits'].items():
    print(f"  {split_name.capitalize():5s}: X={x.shape}, y={y.shape}")

# Plot the RGB composite of the first 3 ICA components
plt.figure(figsize=(8, 12))
plt.imshow(result['rgb_composite'])
plt.title("FastICA False-Color Composite (Components 1, 2, 3)")
plt.axis('off')
plt.show()